In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import warnings

warnings.filterwarnings("ignore")

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.layers import Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

In [ ]:
print("Training Images:", X_train.shape)
print("Training Labels:", y_train.shape)
print("Testing Images:", X_test.shape)
print("Testing Labels:", y_test.shape)

In [ ]:
class_names = [
    "Airplane",
    "Automobile",
    "Bird",
    "Cat",
    "Deer",
    "Dog",
    "Frog",
    "Horse",
    "Ship",
    "Truck"
]

In [ ]:
plt.figure(figsize=(10,10))

for i in range(16):
    plt.subplot(4,4,i+1)
    plt.imshow(X_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis("off")

plt.show()

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

In [ ]:
y_train_cat = to_categorical(y_train,10)
y_test_cat = to_categorical(y_test,10)

In [ ]:
print(X_train.min(), X_train.max())
print(X_train.dtype)

In [ ]:
print(y_train_cat.shape)
print(y_test_cat.shape)

In [ ]:
sample = np.random.randint(0,len(X_train))

plt.imshow(X_train[sample])
plt.title(class_names[np.argmax(y_train_cat[sample])])
plt.axis("off")
plt.show()

In [ ]:
pd.Series(y_train.flatten()).value_counts()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(x=y_train.flatten())

plt.xlabel("Class")

plt.ylabel("Count")

plt.show()

In [ ]:
print("Image Shape:", X_train[0].shape)

In [ ]:
print(class_names)

In [ ]:
print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(X_train[100])

plt.title(class_names[np.argmax(y_train_cat[100])])

plt.axis("off")

plt.show()

In [ ]:
print(np.unique(y_train))

In [ ]:
train_df = pd.DataFrame({
    "Class": y_train.flatten()
})

train_df.head()

In [ ]:
print(train_df["Class"].value_counts())

In [ ]:
print("Dataset Ready for ANN and CNN Training")

In [ ]:
ann = Sequential([
    Flatten(input_shape=(32,32,3)),
    Dense(512,activation="relu"),
    Dense(256,activation="relu"),
    Dense(128,activation="relu"),
    Dense(10,activation="softmax")
])

ann.summary()

In [ ]:
ann.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [ ]:
history_ann = ann.fit(
    X_train,
    y_train_cat,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    callbacks=[early_stop]
)

In [ ]:
ann.evaluate(X_test,y_test_cat)

In [ ]:
ann_pred = ann.predict(X_test)

In [ ]:
ann_pred_classes = np.argmax(ann_pred,axis=1)

In [ ]:
true_classes = y_test.flatten()

In [ ]:
print(classification_report(true_classes,ann_pred_classes,target_names=class_names))

In [ ]:
cm = confusion_matrix(true_classes,ann_pred_classes)

plt.figure(figsize=(8,6))

sns.heatmap(cm,annot=True,fmt="d",cmap="Blues")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_ann.history["accuracy"])

plt.plot(history_ann.history["val_accuracy"])

plt.legend(["Train","Validation"])

plt.title("ANN Accuracy")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_ann.history["loss"])

plt.plot(history_ann.history["val_loss"])

plt.legend(["Train","Validation"])

plt.title("ANN Loss")

plt.show()

In [ ]:
ann_accuracy = ann.evaluate(X_test,y_test_cat,verbose=0)[1]

print(ann_accuracy)

In [ ]:
sample = np.random.randint(0,len(X_test))

plt.imshow(X_test[sample])

plt.title("Actual : "+class_names[true_classes[sample]])

plt.axis("off")

plt.show()

In [ ]:
print("Predicted :",class_names[ann_pred_classes[sample]])

In [ ]:
for i in range(5):

    plt.figure(figsize=(2,2))

    plt.imshow(X_test[i])

    plt.title("Actual : "+class_names[true_classes[i]]+"\nPred : "+class_names[ann_pred_classes[i]])

    plt.axis("off")

    plt.show()

In [ ]:
ann.save("ANN_CIFAR10.keras")

In [ ]:
from google.colab import files

files.download("ANN_CIFAR10.keras")

In [ ]:
print("ANN Test Accuracy :",ann_accuracy)

In [ ]:
print("ANN Model Completed")

In [ ]:
cnn = Sequential([
    Conv2D(32,(3,3),padding="same",activation="relu",input_shape=(32,32,3)),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    Conv2D(64,(3,3),padding="same",activation="relu"),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    Conv2D(128,(3,3),padding="same",activation="relu"),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    Flatten(),

    Dense(256,activation="relu"),
    Dropout(0.5),

    Dense(10,activation="softmax")
])

cnn.summary()

In [ ]:
cnn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

In [ ]:
checkpoint = ModelCheckpoint(
    "Best_CNN.keras",
    monitor="val_accuracy",
    save_best_only=True
)

In [ ]:
history_cnn = cnn.fit(
    X_train,
    y_train_cat,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    callbacks=[early_stop,checkpoint]
)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 144s 224ms/step - accuracy: 0.3761 - loss: 1.7468 - val_accuracy: 0.2890 - val_loss: 3.1330
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 137s 216ms/step - accuracy: 0.5043 - loss: 1.3838 - val_accuracy: 0.4430 - val_loss: 1.9207
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 135s 216ms/step - accuracy: 0.5685 - loss: 1.2227 - val_accuracy: 0.4993 - val_loss: 1.7175
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 138s 221ms/step - accuracy: 0.6114 - loss: 1.1093 - val_accuracy: 0.6253 - val_loss: 1.0471
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - accuracy: 0.6362 - loss: 1.0331

In [ ]:
cnn.evaluate(X_test,y_test_cat)

In [ ]:
cnn_pred = cnn.predict(X_test)

In [ ]:
cnn_pred_classes = np.argmax(cnn_pred,axis=1)

In [ ]:
print(classification_report(
    true_classes,
    cnn_pred_classes,
    target_names=class_names
))

In [ ]:
cm = confusion_matrix(
    true_classes,
    cnn_pred_classes
)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Greens"
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_cnn.history["accuracy"])

plt.plot(history_cnn.history["val_accuracy"])

plt.legend(["Train","Validation"])

plt.title("CNN Accuracy")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_cnn.history["loss"])

plt.plot(history_cnn.history["val_loss"])

plt.legend(["Train","Validation"])

plt.title("CNN Loss")

plt.show()

In [ ]:
cnn_accuracy = cnn.evaluate(
    X_test,
    y_test_cat,
    verbose=0
)[1]

print(cnn_accuracy)

In [ ]:
sample = np.random.randint(
    0,
    len(X_test)
)

plt.imshow(X_test[sample])

plt.title(
    "Actual : "+
    class_names[true_classes[sample]]
)

plt.axis("off")

plt.show()

In [ ]:
print(
    "Predicted :",
    class_names[
        cnn_pred_classes[sample]
    ]
)

In [ ]:
for i in range(5):

    plt.figure(figsize=(2,2))

    plt.imshow(X_test[i])

    plt.title(
        "Actual : "
        + class_names[true_classes[i]]
        + "\nPred : "
        + class_names[cnn_pred_classes[i]]
    )

    plt.axis("off")

    plt.show()

In [ ]:
comparison = pd.DataFrame({

    "Model":[
        "ANN",
        "CNN"
    ],

    "Accuracy":[
        ann_accuracy,
        cnn_accuracy
    ]

})

comparison

In [ ]:
plt.figure(figsize=(6,5))

sns.barplot(
    data=comparison,
    x="Model",
    y="Accuracy"
)

plt.show()

In [ ]:
cnn.save(
    "CNN_CIFAR10.keras"
)

In [ ]:
print("CNN Completed Successfully")

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.2
)

datagen.fit(X_train)

In [ ]:
cnn_aug = models.Sequential([
    layers.Conv2D(32,(3,3),padding="same",activation="relu",input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Conv2D(64,(3,3),padding="same",activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.30),

    layers.Conv2D(128,(3,3),padding="same",activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(128,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.35),

    layers.Flatten(),

    layers.Dense(512,activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(10,activation="softmax")
])

cnn_aug.summary()

In [ ]:
cnn_aug.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stop_aug = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [ ]:
checkpoint_aug = ModelCheckpoint(
    "Best_Augmented_CNN.keras",
    monitor="val_accuracy",
    save_best_only=True
)

In [ ]:
history_aug = cnn_aug.fit(
    datagen.flow(X_train,y_train_cat,batch_size=64),
    validation_data=(X_test,y_test_cat),
    epochs=20,
    callbacks=[early_stop_aug,checkpoint_aug]
)

In [ ]:
cnn_aug.evaluate(X_test,y_test_cat)

In [ ]:
aug_pred = cnn_aug.predict(X_test)

aug_pred_classes = np.argmax(aug_pred,axis=1)

aug_accuracy = cnn_aug.evaluate(
    X_test,
    y_test_cat,
    verbose=0
)[1]

print(aug_accuracy)

In [ ]:
print(classification_report(
    true_classes,
    aug_pred_classes,
    target_names=class_names
))

In [ ]:
cm = confusion_matrix(
    true_classes,
    aug_pred_classes
)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Purples"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_aug.history["accuracy"])

plt.plot(history_aug.history["val_accuracy"])

plt.legend(["Train","Validation"])

plt.title("Augmented CNN Accuracy")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_aug.history["loss"])

plt.plot(history_aug.history["val_loss"])

plt.legend(["Train","Validation"])

plt.title("Augmented CNN Loss")

plt.show()

In [ ]:
comparison = pd.DataFrame({

    "Model":[
        "ANN",
        "CNN",
        "CNN + Augmentation"
    ],

    "Accuracy":[
        ann_accuracy,
        cnn_accuracy,
        aug_accuracy
    ]

})

comparison

In [ ]:
plt.figure(figsize=(7,5))

sns.barplot(
    data=comparison,
    x="Model",
    y="Accuracy"
)

plt.show()

In [ ]:
sample = np.random.randint(0,len(X_test))

plt.figure(figsize=(3,3))

plt.imshow(X_test[sample])

plt.title(
    "Actual : "+class_names[true_classes[sample]]
)

plt.axis("off")

plt.show()

In [ ]:
print(
    "Predicted :",
    class_names[
        aug_pred_classes[sample]
    ]
)

In [ ]:
cnn_aug.save("Final_CIFAR10_Model.keras")

In [ ]:
from google.colab import files

files.download("Final_CIFAR10_Model.keras")

In [ ]:
comparison.sort_values(
    by="Accuracy",
    ascending=False
)

In [ ]:
print("Project Completed Successfully")